# 04 · Find new vacancies (daily job)
DOU + Djinni RSS → `vacancies` → rule prefilter (your ⚙️ Settings) → keyword relevance →
LLM fit score (top N new) → `matches` → optional Telegram message.

Feeds are listed in `config/feeds.yaml`. If serverless can't reach the sites, the job reads the
daily snapshot made by the GitHub Action (`.github/workflows/fetch_feeds.yml`).
Schedule: notebook toolbar → **Schedule** → daily.

In [ ]:
%pip install -q -r ../requirements.txt

In [ ]:
dbutils.library.restartPython()

In [ ]:
import os, sys, json
sys.path.append(os.path.abspath("../src"))

from career_agent import config
from career_agent.bank import read_bank
from career_agent.jobs import (fetch_all, rule_filter, keyword_score, profile_text,
                               score_vacancy, telegram_message, send_telegram)

me = spark.sql("SELECT current_user()").first()[0]
dbutils.widgets.text("user_email", me)
dbutils.widgets.text("max_llm_scored", "20")
dbutils.widgets.text("notify_min_score", "7")
user = dbutils.widgets.get("user_email")
MAX_LLM = int(dbutils.widgets.get("max_llm_scored"))
NOTIFY_MIN = int(dbutils.widgets.get("notify_min_score"))

T_VAC, T_MATCH, T_PREFS = config.table("vacancies"), config.table("matches"), config.table("preferences")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {T_VAC} (
  url STRING, source STRING, title STRING, company STRING, location STRING,
  description STRING, published STRING, first_seen TIMESTAMP)""")
spark.sql(f"""CREATE TABLE IF NOT EXISTS {T_MATCH} (
  user_email STRING, url STRING, source STRING, title STRING, company STRING, location STRING,
  published STRING, score INT, reason STRING, matched ARRAY<STRING>, gaps ARRAY<STRING>,
  notified BOOLEAN, scored_at TIMESTAMP)""")

# Preferences: the user's ⚙️ Settings from the app, else config/preferences.yaml
prefs = config.default_prefs()
if spark.catalog.tableExists(T_PREFS):
    row = spark.sql(f"SELECT prefs FROM {T_PREFS} WHERE user_email = '{user}'").first()
    if row:
        prefs = {**prefs, **json.loads(row["prefs"])}
print("User:", user); print("Preferences:", json.dumps(prefs, ensure_ascii=False))

bank = read_bank(spark)
print(f"Experience bank: {len(bank)} items")

## 1. Fetch feeds

In [ ]:
vacancies, fetch_log = fetch_all()
display(spark.createDataFrame(fetch_log))
print(f"{len(vacancies)} unique vacancies fetched")
if not vacancies:
    dbutils.notebook.exit("No vacancies fetched: check the fetch log (all 'failed' → enable the GitHub Action).")

In [ ]:
from pyspark.sql import functions as F

vac_df = spark.createDataFrame(
    vacancies, "url STRING, source STRING, title STRING, company STRING, location STRING, description STRING, published STRING"
).withColumn("first_seen", F.current_timestamp())
vac_df.createOrReplaceTempView("new_vacancies")
spark.sql(f"""MERGE INTO {T_VAC} t USING new_vacancies s ON t.url = s.url
              WHEN NOT MATCHED THEN INSERT *""")

## 2. Prefilter (rules from Settings + keyword relevance to the bank)

In [ ]:
already = {r.url for r in spark.sql(f"SELECT url FROM {T_MATCH} WHERE user_email = '{user}'").collect()}
candidates, rejected = [], []
for v in vacancies:
    if v["url"] in already:
        continue
    why = rule_filter(v, prefs)
    if why:
        rejected.append({"title": v["title"], "reason": why})
    else:
        candidates.append({**v, "kw": keyword_score(v, bank, prefs)})

candidates.sort(key=lambda v: -v["kw"])
to_score = [v for v in candidates if v["kw"] > 0][:MAX_LLM]
print(f"new: {len(candidates) + len(rejected)} · rejected by rules: {len(rejected)} · "
      f"passed: {len(candidates)} · sent to LLM: {len(to_score)}")
if rejected:
    display(spark.createDataFrame(rejected))

## 3. LLM fit score

In [ ]:
profile = profile_text(bank, prefs)
provider = prefs.get("llm_provider") or "databricks"
model = prefs.get("llm_model") or None

scored = []
for v in to_score:
    try:
        s = score_vacancy(v, profile, llm=provider, model=model)
    except Exception as e:
        print("⚠️ scoring failed:", v["title"], str(e)[:120])
        continue
    scored.append({"user_email": user, "url": v["url"], "source": v["source"], "title": v["title"],
                   "company": v["company"], "location": v["location"], "published": v["published"],
                   "score": s["score"], "reason": s["reason"], "matched": s["matched"], "gaps": s["gaps"],
                   "notified": False})
    print(f"{s['score']:>2}/10  {v['title']}  — {s['reason']}")

In [ ]:
if scored:
    m_df = spark.createDataFrame(
        scored, "user_email STRING, url STRING, source STRING, title STRING, company STRING, location STRING, "
                "published STRING, score INT, reason STRING, matched ARRAY<STRING>, gaps ARRAY<STRING>, notified BOOLEAN"
    ).withColumn("scored_at", F.current_timestamp())
    m_df.createOrReplaceTempView("new_matches")
    spark.sql(f"""MERGE INTO {T_MATCH} t USING new_matches s
                  ON t.user_email = s.user_email AND t.url = s.url
                  WHEN NOT MATCHED THEN INSERT *""")

display(spark.sql(f"""SELECT score, title, company, location, source, reason, gaps, url
                      FROM {T_MATCH} WHERE user_email = '{user}' ORDER BY scored_at DESC, score DESC LIMIT 30"""))

## 4. Telegram (optional)
Needs secret `career-agent/telegram_bot_token` and a chat_id in ⚙️ Settings.

In [ ]:
top = sorted([m for m in scored if m["score"] >= NOTIFY_MIN], key=lambda m: -m["score"])
chat_id = (prefs.get("telegram_chat_id") or "").strip()
if not top:
    print(f"No new matches with score ≥ {NOTIFY_MIN}.")
elif not chat_id:
    print("Telegram chat_id not set in Settings — skipping notification.")
else:
    try:
        token = dbutils.secrets.get(config.SECRET_SCOPE, "telegram_bot_token")
        ok = send_telegram(token, chat_id, telegram_message(top[:10]))
        print("Telegram:", "sent ✅" if ok else "failed (is api.telegram.org reachable from serverless?)")
        if ok:
            urls = ", ".join("'" + m["url"].replace("'", "''") + "'" for m in top[:10])
            spark.sql(f"UPDATE {T_MATCH} SET notified = true WHERE user_email = '{user}' AND url IN ({urls})")
    except Exception as e:
        print("Telegram skipped:", str(e)[:150])